# Track 1 · 10 THINGS-EEG1 prepare — 50 more people, in the THINGS-EEG2 window format

**One job.** Add THINGS-EEG1 (Grootswagers et al. 2022) to the training data: harness-made
windows from 50 people, written in the same format and channel order as the THINGS-EEG2
windows, so `track1_09` trains on both.

Self-contained: upload this one file. Needs `track1_08` to have run (it reads that
notebook's image table, image list and channel list from Drive).

---

## The data

OpenNeuro `ds003825` v1.2.0, **CC0-1.0**; supported by the organisers as an image-task
dataset (`neuralbench/tasks/eeg/image/datasets/grootswagers2022human.yaml`). 50 people, 63
channels at 1 kHz (people 49–50 have 127; the same 63 names are picked), one RSVP
recording each: **22,248 images at 10 per second** (THINGS-EEG2 shows 5 per second), so
each 1-s window overlaps more neighbouring images.

Read as neuralfetch's `Grootswagers2022Human` reads it: events from `events.tsv`, stimulus
time `onset / 1000` s, image named by its `stim` path. The BrainVision file is read with
NumPy (checked against MNE on person 1: max relative difference 8e-8), and preprocessing is
the harness's image pipeline, the same code as `track1_08`.

## What is kept, and what is not

- **The warm-up test concepts are dropped entirely.** THINGS-EEG1 shows all 1,854 THINGS
  concepts, including the 200 the warm-up tests on, and 178 of the 200 exact test images.
  Training on them would inflate the board without real learning, so every trial from those
  200 concepts goes (2,400 per person).
- **Only images already in our target table are kept.** Of the remaining 19,848 trials per
  person, 16,540 show THINGS-EEG2's training images, whose DINOv2 targets we have; the other
  3,308 images would need the full THINGS image database (password-protected) and a new
  embedding pass. Left out for now.
- **A sample per person** (`WINDOWS_PER_PERSON`, default 4,000) to fit Drive: 200,000
  windows, about 3 GB. People are the new axis here; the final is scored on unseen people.
- Channels are put in THINGS-EEG2's order by name; any THINGS-EEG2 channel that
  THINGS-EEG1 lacks (it was referenced to Cz) is zero-filled, as the submission does for a
  different headset.

## Cost

About 41 GB downloaded through temporary disk, one person at a time; about 3 GB on Drive.
Roughly 1.5–2 hours. It resumes: people already on Drive are skipped.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

PEOPLE = list(range(1, 51))
WINDOWS_PER_PERSON = 4000
SEED = 33

# Harness preprocessing: identical to track1_08 (neuralbench defaults + image task).
SFREQ = 120.0
NOTCH = (50.0, 60.0)
BANDPASS = (0.1, 75.0)
TMIN, DURATION = -0.2, 1.0
BASELINE = (-0.2, 0.0)
CLAMP = 20.0

OPENNEURO = 'https://s3.amazonaws.com/openneuro.org/ds003825'
WORK = '/content/things1_work'     # temporary disk; wiped with the session
print('NOTEBOOK VERSION: track1_10 v1')

In [ ]:
%pip install -q mne

import importlib
for mod in ('mne', 'numpy'):
    print(f'{mod:8} {importlib.import_module(mod).__version__}')

In [ ]:
import csv, gc, json, re, shutil, time, urllib.request, warnings
warnings.filterwarnings('ignore')
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np, mne
mne.set_log_level('ERROR')

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
TH2 = ROOT / 'Track1' / 'things_eeg2'
OUT = ROOT / 'Track1' / 'things_eeg1'
WIN = OUT / 'windows'
WIN.mkdir(parents=True, exist_ok=True)
WORK = Path(WORK) if Path('/content').exists() else OUT.parent / '_things1_work'
WORK.mkdir(parents=True, exist_ok=True)

# From track1_08: the channel order, the target table and the image list.
CH = json.loads((TH2 / 'manifest.json').read_text())['channels']
table_keys = np.load(TH2 / 'image_embeddings.npz')['keys'].tolist()
meta = np.load(TH2 / 'image_metadata.npy', allow_pickle=True).item()


def norm(key):
    """THINGS-EEG2 prefixes concepts with a number ('00001_aardvark/...'); EEG1 does not."""
    concept, name = key.replace('\\', '/').split('/')[-2:]
    return re.sub(r'^\d+_', '', concept) + '/' + name


to_table = {norm(k): k for k in table_keys}
test_concepts = {norm(f'{c}/{f}').split('/')[0]
                 for c, f in zip(meta['test_img_concepts'], meta['test_img_files'])}
print(f'{len(CH)} channels, {len(table_keys):,} images in the target table, '
      f'{len(test_concepts)} warm-up test concepts to exclude')

## Step 1 — read, preprocess and window each person

Downloads run one person ahead of processing. Each person's raw files are deleted once
their windows are on Drive.

In [ ]:
BV_DTYPE = {'IEEE_FLOAT_32': '<f4', 'INT_16': '<i2', 'INT_32': '<i4'}
BV_UNIT = {'V': 1.0, 'mV': 1e-3, 'µV': 1e-6, 'uV': 1e-6, 'nV': 1e-9}


def read_brainvision(vhdr):
    """Minimal BrainVision reader in NumPy (multiplexed binary), scaled to volts the way
    MNE does (resolution x unit). Skips the marker file, so no MNE strings are involved.
    Returns (data[n_chans, n_times] float32, channel names, sampling rate)."""
    text = Path(vhdr).read_text(encoding='latin-1')
    get = lambda key: re.search(rf'^{key}=(.*)$', text, re.M).group(1).strip()
    if get('DataOrientation').upper() != 'MULTIPLEXED':
        raise ValueError('only multiplexed BrainVision files are handled')
    n = int(get('NumberOfChannels'))
    sfreq = 1e6 / float(get('SamplingInterval'))
    names, scale = [], []
    for i in range(1, n + 1):
        parts = get(f'Ch{i}').split(',')
        names.append(parts[0])
        res = float(parts[2]) if len(parts) > 2 and parts[2] else 1.0
        unit = parts[3] if len(parts) > 3 and parts[3] else 'µV'
        scale.append(res * BV_UNIT.get(unit, 1e-6))
    raw = np.fromfile(Path(vhdr).with_name(get('DataFile')), dtype=BV_DTYPE[get('BinaryFormat')])
    data = raw.reshape(-1, n).T.astype('float32') * np.array(scale, dtype='float32')[:, None]
    return data, names, sfreq


N_TIMES = int(round(DURATION * SFREQ))
PRE = int(round(-TMIN * SFREQ))
B0, B1 = (int(round((b - TMIN) * SFREQ)) for b in BASELINE)


def harness_windows(data, names, sfreq, onsets):
    """The harness image pipeline (as track1_08): notch, band-pass, resample to 120 Hz,
    RobustScaler per channel over the whole recording, then windows, baseline, clamp."""
    raw = mne.io.RawArray(data.astype('float64'), mne.create_info(names, sfreq, 'eeg'),
                          verbose=False)
    del data
    nyq = sfreq / 2
    freqs = sorted({f for base in NOTCH for f in np.arange(base, min(nyq, 301), base)})
    raw.notch_filter(freqs, phase='zero', filter_length='auto', verbose=False)
    lo, hi = BANDPASS
    raw.filter(lo, hi if hi < nyq else None, verbose=False)
    raw.resample(SFREQ, verbose=False)
    x = raw.get_data()
    del raw
    med = np.median(x, axis=1, keepdims=True)
    iqr = np.subtract(*np.percentile(x, [75, 25], axis=1))[:, None]
    iqr[iqr == 0] = 1.0
    x = ((x - med) / iqr).astype('float32')
    s = np.round(np.asarray(onsets) * SFREQ).astype(int) - PRE
    ok = (s >= 0) & (s + N_TIMES <= x.shape[1])
    X = np.stack([x[:, i:i + N_TIMES] for i in s[ok]])
    X = X - X[:, :, B0:B1].mean(axis=2, keepdims=True)
    return np.clip(X, -CLAMP, CLAMP), ok


def fetch(k):
    """Download one person's header, binary and events to temporary disk."""
    sub = f'sub-{k:02d}'
    d = WORK / sub
    d.mkdir(exist_ok=True)
    for name in (f'{sub}_task-rsvp_eeg.vhdr', f'{sub}_task-rsvp_events.tsv', f'{sub}_task-rsvp_eeg.eeg'):
        dst = d / name
        if dst.exists():
            continue
        for attempt in range(5):
            try:
                with urllib.request.urlopen(f'{OPENNEURO}/{sub}/eeg/{name}', timeout=300) as r, \
                     open(dst.with_name(name + '.part'), 'wb') as f:
                    shutil.copyfileobj(r, f, 32 << 20)
                dst.with_name(name + '.part').rename(dst)
                break
            except Exception as e:
                if attempt == 4:
                    raise RuntimeError(f'{sub}/{name}: {e}')
                time.sleep(10 * (attempt + 1))
    return d


def do_person(k, folder):
    sub = f'sub-{k:02d}'
    out = WIN / f'{sub}_things1_train.npz'
    with open(folder / f'{sub}_task-rsvp_events.tsv', newline='') as f:
        rows = list(csv.DictReader(f, delimiter='\t'))
    keys = [norm(r['stim']) for r in rows]
    onsets = np.array([float(r['onset']) / 1000.0 for r in rows])   # neuralfetch: onset / 1000
    test = np.array([k_.split('/')[0] in test_concepts for k_ in keys])
    known = np.array([k_ in to_table for k_ in keys])
    eligible = np.where(~test & known)[0]
    rng = np.random.default_rng(SEED + k)
    pick = np.sort(rng.choice(eligible, min(WINDOWS_PER_PERSON, len(eligible)), replace=False))

    data, names, sfreq = read_brainvision(folder / f'{sub}_task-rsvp_eeg.vhdr')
    lookup = {n.casefold(): i for i, n in enumerate(names)}
    have = [c for c in CH if c.casefold() in lookup]
    missing = [c for c in CH if c.casefold() not in lookup]
    sel = data[[lookup[c.casefold()] for c in have]]                 # only the channels we use
    del data
    Xh, ok = harness_windows(sel, have, sfreq, onsets[pick])
    del sel
    X = np.zeros((len(Xh), len(CH), N_TIMES), dtype='float16')
    for j, c in enumerate(have):
        X[:, CH.index(c)] = Xh[:, j]
    info = dict(person=k, recorded_channels=len(names), matched=len(have), zero_filled=missing,
                trials=len(rows), dropped_test_concepts=int(test.sum()),
                dropped_unknown_images=int((~test & ~known).sum()), eligible=int(len(eligible)),
                kept=int(ok.sum()))
    np.savez(out, X=X, keys=np.array([to_table[keys[i]] for i in pick[ok]]), subject=100 + k,
             session=1, split='train', channels=np.array(CH), info=json.dumps(info))
    shutil.rmtree(folder, ignore_errors=True)
    return info


todo = [k for k in PEOPLE if not (WIN / f'sub-{k:02d}_things1_train.npz').exists()]
print(f'{len(PEOPLE) - len(todo)} people already on Drive, {len(todo)} to do')
t0 = time.time()
failed = {}
with ThreadPoolExecutor(1) as pool:                     # downloads one person ahead
    nxt = pool.submit(fetch, todo[0]) if todo else None
    for i, k in enumerate(todo):
        try:
            folder = nxt.result()
        except Exception as e:
            failed[k] = repr(e); print(f'  FAILED download sub-{k:02d}: {e}')
            folder = None
        nxt = pool.submit(fetch, todo[i + 1]) if i + 1 < len(todo) else None
        if folder is None:
            continue
        try:
            info = do_person(k, folder)
            print(f'  sub-{k:02d}: {info["trials"]:,} trials, {info["dropped_test_concepts"]:,} from test '
                  f'concepts dropped, {info["eligible"]:,} eligible -> kept {info["kept"]:,}; channels '
                  f'{info["matched"]}/{len(CH)} matched of {info["recorded_channels"]} recorded, zero-filled '
                  f'{info["zero_filled"] or "none"}   {time.time() - t0:.0f}s')
        except Exception as e:
            failed[k] = repr(e); print(f'  FAILED sub-{k:02d}: {e}')
        gc.collect()
print(f'\n{len(todo) - len(failed)} people done, {len(failed)} failed, {time.time() - t0:.0f}s')
if failed:
    print('Rerun this cell: finished people are kept and only the failures are retried.')

## Step 2 — checks and manifest

In [ ]:
files = sorted(WIN.glob('sub-*_things1_train.npz'))
n, problems, people, zero = 0, [], [], {}
table = set(table_keys)
for f in files:
    z = np.load(f, allow_pickle=True)
    info = json.loads(str(z['info']))
    keys = z['keys'].tolist()
    n += len(keys); people.append(info['person'])
    for c in info['zero_filled']:
        zero[c] = zero.get(c, 0) + 1
    if z['X'].shape[1:] != (len(CH), N_TIMES):
        problems.append(f'{f.name}: shape {z["X"].shape}')
    if list(z['channels']) != CH:
        problems.append(f'{f.name}: channel order')
    if set(keys) - table:
        problems.append(f'{f.name}: images without a target')
    if any(norm(k_).split('/')[0] in test_concepts for k_ in keys):
        problems.append(f'{f.name}: a warm-up test concept slipped in')
    if not np.isfinite(z['X'].astype('float32')).all():
        problems.append(f'{f.name}: non-finite values')
print(f'{len(files)} people, {n:,} windows; zero-filled channels (people): {zero or "none"}')
print('checks:', 'all passed' if not problems else problems[:10])
(OUT / 'manifest.json').write_text(json.dumps(dict(
    dataset='THINGS-EEG1 (grootswagers2022human, OpenNeuro ds003825 v1.2.0)', licence='CC0-1.0',
    people=sorted(people), windows=n, windows_per_person=WINDOWS_PER_PERSON, seed=SEED,
    excluded='all trials of the 200 THINGS-EEG2 test concepts; images without a target',
    channels=CH, sfreq=SFREQ, n_times=N_TIMES, subject_ids='100 + person',
    preprocessing=dict(notch=NOTCH, bandpass=BANDPASS, scaler='RobustScaler',
                       baseline=BASELINE, clamp=CLAMP, window=(TMIN, TMIN + DURATION))),
    indent=1))
size = sum(f.stat().st_size for f in files) / 1e9
print(f'manifest -> {OUT / "manifest.json"}   windows on Drive: {size:.2f} GB')
print('READY: in track1_09 set EXTRA_SOURCES = ["things_eeg1"]' if files and not problems and not failed
      else 'NOT READY: see above')